# Zolai Dataset Cleaner — Kaggle Notebook

This notebook cleans and standardizes the Zolai-focused dataset for training.
It processes:
- Bible parallel pairs (Zolai ↔ English)
- Zolai dictionary entries (from Tongdot)
- Grammar instruction pairs
- Zolai text corpus (Simbu, articles)

**Output:** Clean JSONL files ready for model training.

In [ ]:
# First cell: check internet and install deps
import os
import socket

try:
    socket.create_connection(("8.8.8.8", 53), timeout=3)
    HAS_INTERNET = True
except OSError:
    HAS_INTERNET = False

print(f"Internet: {'Connected' if HAS_INTERNET else 'Offline'}")

if HAS_INTERNET:
    !pip install pandas jsonlines

In [ ]:
import json
import re
import hashlib
from pathlib import Path
from collections import Counter

import pandas as pd

# Paths — adjust if your Kaggle dataset mount point differs
INPUT_DIR = Path("/kaggle/input/zolai-dataset")
OUTPUT_DIR = Path("/kaggle/working")

# Fallback: if running locally, use this path
# INPUT_DIR = Path(r"data/zolai-focused")
# OUTPUT_DIR = Path(r"data/cleaned")
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Cleaning Utilities

In [ ]:
def clean_zolai_text(text: str) -> str:
    """Clean Zolai text: normalize whitespace, fix common issues."""
    if not text or not isinstance(text, str):
        return ""
    
    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Fix common OCR/scraping artifacts
    text = text.replace('\u2019', "'")   # curly apostrophe
    text = text.replace('\u2018', "'")   # curly single quote
    text = text.replace('\u201c', '"')   # curly double quote open
    text = text.replace('\u201d', '"')   # curly double quote close
    text = text.replace('\u2013', '-')   # en dash
    text = text.replace('\u2014', '--')  # em dash
    text = text.replace('\u2026', '...') # ellipsis
    text = text.replace('\u00ad', '')    # soft hyphen
    text = text.replace('\ufeff', '')    # BOM
    
    # Fix spacing around punctuation
    text = re.sub(r'\s+([,.;:!?])', r'\1', text)
    text = re.sub(r'([,.;:!?])\s*', r'\1 ', text).strip()
    
    return text


def text_hash(text: str) -> str:
    """MD5 hash for deduplication."""
    return hashlib.md5(text.encode("utf-8")).hexdigest()


def has_burmese(text: str) -> bool:
    """Check if text contains Burmese script."""
    return any('\u1000' <= c <= '\u109F' for c in text)


def has_html_tags(text: str) -> bool:
    """Check if text contains HTML tags."""
    return bool(re.search(r'<[^>]+>', text))


def strip_html(text: str) -> str:
    """Remove all HTML tags from text."""
    return re.sub(r'<[^>]+>', '', text)

## 2. Clean Bible Parallel Pairs

In [ ]:
def clean_bible_pairs(input_path: Path, output_path: Path):
    """Clean Bible Zolai-English parallel pairs."""
    if not input_path.exists():
        print(f"SKIP: {input_path} not found")
        return
    
    count = 0
    dupes = 0
    seen = set()
    
    with open(input_path, "r", encoding="utf-8") as f_in, \
         open(output_path, "w", encoding="utf-8") as f_out:
        
        for line in f_in:
            line = line.strip()
            if not line:
                continue
            
            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                continue
            
            source = clean_zolai_text(record.get("source", ""))
            target = clean_zolai_text(record.get("target", ""))
            
            if not source or not target:
                continue
            
            # Dedup by source text
            h = text_hash(source)
            if h in seen:
                dupes += 1
                continue
            seen.add(h)
            
            clean_record = {
                "source": source,
                "target": target,
                "verse_id": record.get("verse_id", ""),
                "src_lang": "zolai",
                "tgt_lang": "english",
                "type": "translation_pair"
            }
            
            f_out.write(json.dumps(clean_record, ensure_ascii=False) + "\n")
            count += 1
    
    print(f"Bible pairs: {count} cleaned, {dupes} duplicates removed")


# Find and clean Bible pairs
bible_files = list(INPUT_DIR.rglob("*bible*english*pairs*.jsonl")) + \
              list(INPUT_DIR.rglob("*bible_parallel*.jsonl"))

for bf in bible_files:
    print(f"Processing: {bf.name}")
    out_name = f"cleaned_{bf.name}"
    clean_bible_pairs(bf, OUTPUT_DIR / out_name)

## 3. Clean Dictionary Entries

In [ ]:
def clean_dictionary(input_path: Path, output_path: Path):
    """Extract and clean Zolai dictionary entries."""
    if not input_path.exists():
        print(f"SKIP: {input_path} not found")
        return
    
    count = 0
    zolai_langs = {"tedim", "haka", "laizo/falam", "matu", "zolai", "zomi"}
    
    with open(input_path, "r", encoding="utf-8") as f_in, \
         open(output_path, "w", encoding="utf-8") as f_out:
        
        for line in f_in:
            line = line.strip()
            if not line:
                continue
            
            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                continue
            
            query = record.get("query", "")
            
            for r in record.get("results", []):
                lang = r.get("language", "").lower()
                if lang not in zolai_langs:
                    continue
                
                headword = r.get("headword", query)
                translation = clean_zolai_text(r.get("translation", ""))
                
                if not translation:
                    continue
                
                clean_record = {
                    "headword": headword,
                    "translation": translation,
                    "language": lang,
                    "type": "dictionary",
                    "source": "tongdot"
                }
                
                f_out.write(json.dumps(clean_record, ensure_ascii=False) + "\n")
                count += 1
    
    print(f"Dictionary: {count} Zolai entries extracted")


# Find and clean dictionary
dict_files = list(INPUT_DIR.rglob("*tongdot*zolai*.jsonl")) + \
             list(INPUT_DIR.rglob("*dictionary*.jsonl"))

for df in dict_files:
    print(f"Processing: {df.name}")
    out_name = f"cleaned_{df.name}"
    clean_dictionary(df, OUTPUT_DIR / out_name)

## 4. Clean Grammar Instructions

In [ ]:
def clean_grammar(input_path: Path, output_path: Path):
    """Clean grammar instruction pairs."""
    if not input_path.exists():
        print(f"SKIP: {input_path} not found")
        return
    
    count = 0
    
    with open(input_path, "r", encoding="utf-8") as f_in, \
         open(output_path, "w", encoding="utf-8") as f_out:
        
        for line in f_in:
            line = line.strip()
            if not line:
                continue
            
            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                continue
            
            instruction = clean_zolai_text(record.get("instruction", ""))
            inp = clean_zolai_text(record.get("input", ""))
            output = clean_zolai_text(record.get("output", ""))
            
            if not instruction or not output:
                continue
            
            clean_record = {
                "instruction": instruction,
                "input": inp,
                "output": output,
                "type": "instruction",
                "source": "grammar"
            }
            
            f_out.write(json.dumps(clean_record, ensure_ascii=False) + "\n")
            count += 1
    
    print(f"Grammar: {count} instruction pairs cleaned")


# Find and clean grammar
grammar_files = list(INPUT_DIR.rglob("*grammar*.jsonl"))

for gf in grammar_files:
    print(f"Processing: {gf.name}")
    out_name = f"cleaned_{gf.name}"
    clean_grammar(gf, OUTPUT_DIR / out_name)

## 5. Clean Zolai Text Corpus

In [ ]:
def clean_text_corpus(input_path: Path, output_path: Path):
    """Clean Zolai text corpus (Simbu, articles, etc.)."""
    if not input_path.exists():
        print(f"SKIP: {input_path} not found")
        return
    
    count = 0
    skipped = 0
    seen = set()
    
    with open(input_path, "r", encoding="utf-8") as f_in, \
         open(output_path, "w", encoding="utf-8") as f_out:
        
        for line in f_in:
            line = line.strip()
            if not line:
                continue
            
            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                continue
            
            # Handle different record formats
            text = record.get("text", "")
            if not text:
                # Try content field (for articles)
                text = record.get("content", record.get("title", ""))
            
            # Strip HTML if present
            if has_html_tags(text):
                text = strip_html(text)
            
            text = clean_zolai_text(text)
            
            if not text or len(text) < 10:
                skipped += 1
                continue
            
            # Dedup
            h = text_hash(text)
            if h in seen:
                continue
            seen.add(h)
            
            clean_record = {
                "text": text,
                "source": record.get("source", "unknown"),
                "type": record.get("type", "text"),
                "language": "zolai"
            }
            
            f_out.write(json.dumps(clean_record, ensure_ascii=False) + "\n")
            count += 1
    
    print(f"Text corpus: {count} cleaned, {skipped} too short")


# Find and clean text corpus
text_files = list(INPUT_DIR.rglob("*simbu*.jsonl")) + \
             list(INPUT_DIR.rglob("*zolai*combined*.jsonl")) + \
             list(INPUT_DIR.rglob("*zolai*zolai*.jsonl"))

for tf in text_files:
    print(f"Processing: {tf.name}")
    out_name = f"cleaned_{tf.name}"
    clean_text_corpus(tf, OUTPUT_DIR / out_name)

## 6. Merge All Cleaned Data

In [ ]:
def merge_cleaned_files(output_dir: Path, merged_path: Path):
    """Merge all cleaned JSONL files into one dataset."""
    cleaned_files = list(output_dir.glob("cleaned_*.jsonl"))
    
    total = 0
    type_counts = Counter()
    
    with open(merged_path, "w", encoding="utf-8") as f_out:
        for cf in sorted(cleaned_files):
            with open(cf, "r", encoding="utf-8") as f_in:
                for line in f_in:
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        record = json.loads(line)
                        type_counts[record.get("type", "unknown")] += 1
                        f_out.write(line + "\n")
                        total += 1
                    except json.JSONDecodeError:
                        continue
    
    print(f"\nMerged dataset: {total} total records")
    print("By type:")
    for t, c in type_counts.most_common():
        print(f"  {t}: {c}")


merge_cleaned_files(OUTPUT_DIR, OUTPUT_DIR / "zolai_cleaned_merged.jsonl")

## 7. Validation

In [ ]:
# Validate output JSONL
merged = OUTPUT_DIR / "zolai_cleaned_merged.jsonl"
if merged.exists():
    valid = 0
    invalid = 0
    with open(merged, "r", encoding="utf-8") as f:
        for line in f:
            try:
                json.loads(line)
                valid += 1
            except json.JSONDecodeError:
                invalid += 1
    
    print(f"Validation: {valid} valid, {invalid} invalid lines")
    
    # Show sample records
    print("\nSample records:")
    with open(merged, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= 3:
                break
            record = json.loads(line)
            print(json.dumps(record, ensure_ascii=False, indent=2)[:300])
            print("---")